# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

In [1]:
%pip install evaluate rouge_score


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /mimer/NOBACKUP/groups/naiss2025-23-515/yingyi/venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [2]:
import torch
SEED = 101
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 400

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [3]:
from datasets import load_dataset
from datasets import DatasetDict

smoltalk = load_dataset("HuggingFaceTB/smoltalk", 'all')

In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [4]:
smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

In [5]:
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 400
    })
})

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

In [6]:
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

In [23]:
def format_input_output(example):
    # `messages` is a list of messages, each with a `content` string and a `role`.
    messages = example['messages']

    # TODO: implement this
    system_message = None
    user_message = None
    assistant_message = None

    for message in messages:
        if message["role"] == "system":
            system_message = message["content"]
        elif message["role"] == "user":
            user_message = message["content"]
        elif message["role"] == "assistant":
            assistant_message = message["content"]

    if system_message is not None:
        prompt = f"System: {system_message}\nUser: {user_message}\nAssistant:"
    else:
        prompt = f"User: {user_message}\nAssistant:"

    response = f"\n{assistant_message}"

    return {"prompt": prompt, "response": response}

Apply the function you implemented to the dataset as a whole.

In [24]:
ds_sft = smoltalk_simplified.map(format_input_output)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Then verify that the dataset now contains the new fields you created.

In [25]:
ds_sft['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': "System: You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.\nUser: Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.\nAssistant:",
 'response': '\nThe chef made more food after the restaurant ran out.'}

In [26]:
print(ds_sft["train"][0]["prompt"])
print(ds_sft["train"][0]["response"])

System: You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.
User: Rearrange this sentence to make it easy to understand:
The restaurant ran out of food, so the chef made some more.
Assistant:

The chef made more food after the restaurant ran out.


### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

In [27]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [28]:
def tokenize_helper(example):
    prompt = example['prompt']     # Created in the previous step
    response = example['response'] # Created in the previous step

    # TODO: your work goes here.
    # Tokenize prompt and response separately
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    response_ids = tokenizer.encode(response, add_special_tokens=False)

    # Concatenate them to form the full input
    input_ids = prompt_ids + response_ids

    # Attend to all real tokens
    attention_mask = [1] * len(input_ids)

    # Mask the prompt tokens with -100, keep response tokens as labels
    labels = [-100] * len(prompt_ids) + response_ids

    return {
        "input_ids": input_ids, # Input token ids of the prompt and response
        "attention_mask": attention_mask, # Input token ids of the prompt and response
        "labels": labels, # Output token ids of the prompt (masked) and response
    }

In [30]:
tokenized_ds_sft = ds_sft.map(tokenize_helper)
example = tokenized_ds_sft["train"][0]

print(len(example["input_ids"]))
print(len(example["attention_mask"]))
print(len(example["labels"]))

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

76
76
76


As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.


## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [31]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id

    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

In [32]:
import evaluate

class RougeMetricComputer:
    """
    Stateful metric for batch_eval_metrics=True.

    It:
      - accumulates predictions and references across batches
      - computes ROUGE-L once at the end (compute_result=True)
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        """Accumulate predictions and compute at the end."""

        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        # Collect decoded answer-span text from each example in the batch
        for p, lbl in zip(pred_ids, labels):
            mask = lbl != -100
            if mask.sum() == 0:
                continue

            ref_ids = lbl[mask]
            pred_ids_filtered = p[mask]

            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(
                pred_ids_filtered, skip_special_tokens=True,
                eos_token_id=self.tokenizer.vocab['<|im_end|>']
            )

            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        # Only compute at the very end of eval
        if compute_result:
            if len(self.all_references) > 0:
                scores = self.rouge.compute(
                    predictions=self.all_predictions,
                    references=self.all_references,
                )

                # Clear accumulated data for next eval call
                self.all_predictions = []
                self.all_references = []
                return {"rougeL": scores["rougeL"]}
            else:
                return {}
        else:
            return {}

compute_metrics = RougeMetricComputer(tokenizer)


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [33]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

In [36]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import time
import json

print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL")
print("=" * 80)

model_name_or_path = MODEL_NAME

pretrained_model = AutoModelForCausalLM.from_pretrained(model_name_or_path).to("cuda")

pretrained_eval_args = TrainingArguments(
    eval_strategy="no",
    per_device_eval_batch_size=1,
    bf16=True, fp16=False, # This may need to be changed, depending on the model you selected
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

pretrained_eval_loss = float(pretrained_eval_metrics["eval_loss"])
pretrained_rougeL = pretrained_eval_metrics.get("eval_rougeL", None)

print("\nPRETRAINED EVAL METRICS:")
print(json.dumps(pretrained_eval_metrics, indent=2))


EVALUATING PRETRAINED MODEL


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]


PRETRAINED EVAL METRICS:
{
  "eval_loss": 2.0405385494232178,
  "eval_model_preparation_time": 0.0038,
  "eval_rougeL": 0.6040669829910597,
  "eval_runtime": 16.5403,
  "eval_samples_per_second": 24.183,
  "eval_steps_per_second": 24.183,
  "epoch": 0
}



## Part 3: Supervised fine-tuning



### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

In [37]:

baseline_training_args = TrainingArguments(
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=True, fp16=False, # This may need to be changed, depending on the model you selected
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

base_model = AutoModelForCausalLM.from_pretrained(model_name_or_path).to("cuda")

baseline_trainer = make_trainer(base_model, baseline_training_args)

# TODO: train and evaluate the model.

print("\n" + "=" * 80)
print("TRAINING FULL SFT MODEL")
print("=" * 80)

t0 = time.perf_counter()

baseline_train_result = baseline_trainer.train()

baseline_train_time = time.perf_counter() - t0

print("\n" + "=" * 80)
print("EVALUATING FULL SFT MODEL")
print("=" * 80)

t0 = time.perf_counter()

baseline_eval_metrics = baseline_trainer.evaluate()

baseline_eval_time = time.perf_counter() - t0

baseline_eval_loss = float(baseline_eval_metrics["eval_loss"])
baseline_rougeL = baseline_eval_metrics.get("eval_rougeL", None)

print("\nFULL SFT EVAL METRICS:")
print(json.dumps(baseline_eval_metrics, indent=2))

print(f"\nFull SFT train time: {baseline_train_time:.2f} seconds")
print(f"Full SFT eval time: {baseline_eval_time:.2f} seconds")
print(f"Full SFT eval loss: {baseline_eval_loss}")
print(f"Full SFT ROUGE-L: {baseline_rougeL}")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


TRAINING FULL SFT MODEL


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': '1.409', 'grad_norm': '4.469', 'learning_rate': '3.001e-05', 'epoch': '0.4'}
{'loss': '1.279', 'grad_norm': '9', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': '1.311', 'eval_rougeL': '0.645', 'eval_runtime': '15.94', 'eval_samples_per_second': '25.09', 'eval_steps_per_second': '25.09', 'epoch': '1'}
{'train_runtime': '406.3', 'train_samples_per_second': '12.31', 'train_steps_per_second': '12.31', 'train_loss': '1.329', 'epoch': '1'}

EVALUATING FULL SFT MODEL


  0%|          | 0/400 [00:00<?, ?it/s]


FULL SFT EVAL METRICS:
{
  "eval_loss": 1.3106423616409302,
  "eval_rougeL": 0.645029605331768,
  "eval_runtime": 15.9125,
  "eval_samples_per_second": 25.137,
  "eval_steps_per_second": 25.137,
  "epoch": 1.0
}

Full SFT train time: 406.53 seconds
Full SFT eval time: 15.92 seconds
Full SFT eval loss: 1.3106423616409302
Full SFT ROUGE-L: 0.645029605331768


### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [38]:
def num_trainable_parameters(model):
    """Count number of trainable parameters.

    Args:
        model: A PyTorch module.
    """
    # TODO: Add your code here
    
    trainable_params = 0

    for param in model.parameters():
        if param.requires_grad:
            trainable_params += param.numel()

    return trainable_params

In [40]:
full_sft_trainable_params = num_trainable_parameters(base_model)

print(f"Full SFT trainable parameters: {full_sft_trainable_params:,}")

Full SFT trainable parameters: 134,515,008


Apply this function to the SFT-trained model and check that the result makes sense.

## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

In [56]:
import torch.nn as nn

def extract_lora_targets(model):
  # TODO: Add your code here
    target_suffixes = ["q_proj", "k_proj", "v_proj", "o_proj"]
    lora_targets = {}

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and any(name.endswith(suffix) for suffix in target_suffixes):
            lora_targets[name] = module

    return lora_targets

In [57]:
lora_targets = extract_lora_targets(base_model)

print(f"Number of LoRA target layers: {len(lora_targets)}")

for name, layer in list(lora_targets.items())[:8]:
    print(name, layer)

Number of LoRA target layers: 120
model.layers.0.self_attn.q_proj Linear(in_features=576, out_features=576, bias=False)
model.layers.0.self_attn.k_proj Linear(in_features=576, out_features=192, bias=False)
model.layers.0.self_attn.v_proj Linear(in_features=576, out_features=192, bias=False)
model.layers.0.self_attn.o_proj Linear(in_features=576, out_features=576, bias=False)
model.layers.1.self_attn.q_proj Linear(in_features=576, out_features=576, bias=False)
model.layers.1.self_attn.k_proj Linear(in_features=576, out_features=192, bias=False)
model.layers.1.self_attn.v_proj Linear(in_features=576, out_features=192, bias=False)
model.layers.1.self_attn.o_proj Linear(in_features=576, out_features=576, bias=False)


We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [45]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:


In [58]:
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, W, r, alpha):
        super().__init__()
        # TODO: Add your code here
        
        self.W = W
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Freeze the original linear layer
        for param in self.W.parameters():
            param.requires_grad = False

        in_features = W.in_features
        out_features = W.out_features

        # LoRA low-rank matrices
        self.lora_A = nn.Linear(in_features, r, bias=False)
        self.lora_B = nn.Linear(r, out_features, bias=False)

        # Initialize LoRA parameters
        nn.init.kaiming_uniform_(self.lora_A.weight, a=5**0.5)
        nn.init.zeros_(self.lora_B.weight)

        # Put LoRA parameters on the same device and dtype as W
        self.lora_A.to(device=W.weight.device, dtype=W.weight.dtype)
        self.lora_B.to(device=W.weight.device, dtype=W.weight.dtype)

    def forward(self, x):
        # TODO: Add your code here
        
        original_output = self.W(x)
        lora_output = self.lora_B(self.lora_A(x)) * self.scaling

        return original_output + lora_output

In [59]:
test_targets = extract_lora_targets(base_model)
name, linear_layer = next(iter(test_targets.items()))

print(name)
print(linear_layer)

lora_layer = LoRALayer(linear_layer, r=8, alpha=16)

print(lora_layer)

model.layers.0.self_attn.q_proj
Linear(in_features=576, out_features=576, bias=False)
LoRALayer(
  (W): Linear(in_features=576, out_features=576, bias=False)
  (lora_A): Linear(in_features=576, out_features=8, bias=False)
  (lora_B): Linear(in_features=8, out_features=576, bias=False)
)


Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.


In [60]:
def replace_lora_targets(model, r=8, alpha=16):
    # First extract all target names.
    # We store names first because we should not modify modules while iterating over named_modules().
    lora_targets = extract_lora_targets(model)

    for name, module in lora_targets.items():
        parent_name, child_name = name.rsplit(".", 1)

        parent_module = model.get_submodule(parent_name)

        lora_layer = LoRALayer(module, r=r, alpha=alpha)

        setattr(parent_module, child_name, lora_layer)

    return model

In [61]:
lora_model = AutoModelForCausalLM.from_pretrained(model_name_or_path).to(DEVICE)

for param in lora_model.parameters():
    param.requires_grad = False

lora_model = replace_lora_targets(
    lora_model,
    r=8,
    alpha=16,
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [63]:
lora_trainable_params = num_trainable_parameters(lora_model)
lora_total_params = sum(p.numel() for p in lora_model.parameters())

print(f"LoRA trainable parameters: {lora_trainable_params:,}")
print(f"Total parameters: {lora_total_params:,}")
print(f"Percentage trainable: {100 * lora_trainable_params / lora_total_params:.4f}%")

LoRA trainable parameters: 921,600
Total parameters: 135,436,608
Percentage trainable: 0.6805%


In [64]:
lora_training_args = TrainingArguments(
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=True,
    fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

lora_trainer = make_trainer(lora_model, lora_training_args)

print("\n" + "=" * 80)
print("TRAINING LORA MODEL")
print("=" * 80)

t0 = time.perf_counter()

lora_train_result = lora_trainer.train()

lora_train_time = time.perf_counter() - t0

print("\n" + "=" * 80)
print("EVALUATING LORA MODEL")
print("=" * 80)

t0 = time.perf_counter()

lora_eval_metrics = lora_trainer.evaluate()

lora_eval_time = time.perf_counter() - t0

lora_eval_loss = float(lora_eval_metrics["eval_loss"])
lora_rougeL = lora_eval_metrics.get("eval_rougeL", None)

print("\nLORA EVAL METRICS:")
print(json.dumps(lora_eval_metrics, indent=2))

print(f"\nLoRA train time: {lora_train_time:.2f} seconds")
print(f"LoRA eval time: {lora_eval_time:.2f} seconds")
print(f"LoRA eval loss: {lora_eval_loss}")
print(f"LoRA ROUGE-L: {lora_rougeL}")
print(f"LoRA trainable parameters: {lora_trainable_params:,}")


TRAINING LORA MODEL


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': '1.498', 'grad_norm': '0.6406', 'learning_rate': '3.001e-05', 'epoch': '0.4'}
{'loss': '1.365', 'grad_norm': '1.086', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': '1.394', 'eval_rougeL': '0.6345', 'eval_runtime': '19.81', 'eval_samples_per_second': '20.19', 'eval_steps_per_second': '20.19', 'epoch': '1'}
{'train_runtime': '491.1', 'train_samples_per_second': '10.18', 'train_steps_per_second': '10.18', 'train_loss': '1.415', 'epoch': '1'}

EVALUATING LORA MODEL


  0%|          | 0/400 [00:00<?, ?it/s]


LORA EVAL METRICS:
{
  "eval_loss": 1.393972635269165,
  "eval_rougeL": 0.634497543132881,
  "eval_runtime": 19.7915,
  "eval_samples_per_second": 20.211,
  "eval_steps_per_second": 20.211,
  "epoch": 1.0
}

LoRA train time: 491.37 seconds
LoRA eval time: 19.82 seconds
LoRA eval loss: 1.393972635269165
LoRA ROUGE-L: 0.634497543132881
LoRA trainable parameters: 921,600


### 🎓&nbsp; Task 4.4: Qualitative inspection

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

In [67]:
def build_prompt(user_message, system_message=None):
    if system_message is not None:
        return f"System: {system_message}\nUser: {user_message}\nAssistant:"
    else:
        return f"User: {user_message}\nAssistant:"


def generate_response(model, prompt, max_new_tokens=80):
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # Stop when the model starts generating a new conversation turn
    for stop_word in ["\nUser:", "\nSystem:", "\nAssistant:"]:
        if stop_word in text:
            text = text.split(stop_word)[0]

    return text.strip()

test_examples = [
    {
        "system": "You are a helpful assistant.",
        "user": "Explain what machine learning is in one short sentence."
    },
    {
        "system": "You are an AI rewriting assistant.",
        "user": "Rewrite this sentence to make it clearer: The movie was not bad, but it was not very exciting either."
    },
    {
        "system": "You are a helpful assistant.",
        "user": "Give me three tips for studying for an exam."
    }
]


# Get the full SFT model from the variable names used earlier in your notebook
if "base_model" in globals():
    full_sft_model = base_model
elif "baseline_trainer" in globals():
    full_sft_model = baseline_trainer.model
else:
    raise ValueError("Full SFT model is not available. You need to rerun Task 3.1.")


models_to_compare = {
    "Pretrained": pretrained_model,
    "Full SFT": full_sft_model,
    "LoRA": lora_model,
}


for i, example in enumerate(test_examples):
    prompt = build_prompt(
        user_message=example["user"],
        system_message=example["system"],
    )

    print("=" * 100)
    print(f"Example {i + 1}")
    print("-" * 100)
    print("PROMPT:")
    print(prompt)

    for model_name, model in models_to_compare.items():
        response = generate_response(model, prompt)

        print("\n" + model_name + " response:")
        print(response)

Example 1
----------------------------------------------------------------------------------------------------
PROMPT:
System: You are a helpful assistant.
User: Explain what machine learning is in one short sentence.
Assistant:

Pretrained response:
Machine learning is a way to make computers do things that humans do.

Full SFT response:
Machine learning is a type of artificial intelligence that helps computers learn from data and make predictions or decisions without being explicitly programmed.

LoRA response:
Machine learning is a type of artificial intelligence that helps computers learn from data and make predictions or decisions without being explicitly programmed.

Example:
Imagine you are a machine learning expert who has been trained to recognize different types of cats. You can now use your skills to identify which cats are likely to be happy and which ones are likely to be sad.

Example:
Let's say
Example 2
-------------------------------------------------------------------